In [ ]:
import pandas as pd
from pathlib import Path
import geopandas as gpd
import numpy as np


In [ ]:
base_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers")
results_path = base_path / "dphil_paper_2/results/bcr_mca_results"

In [ ]:
jamaica_boundary_path = base_path / "dphil_common_cross_cutting/common_incoming_data/boundaries/jamaica.gpkg"
jamaica_boundary = gpd.read_file(jamaica_boundary_path)
print(jamaica_boundary.crs)

In [ ]:
catchments = gpd.read_file(base_path / "dphil_paper_2/processed_data/major_river_catchments/major_basins_plus_coastal_unionized_final.gpkg")[["catchment_uid","geometry"]]

In [ ]:
# Connectivity ranks
connectivity = (
    pd.read_csv(
        base_path / "dphil_paper_2/results/connectivity_results/catchment_connectivity_norm_pct_all.csv",
        usecols=["catchment_uid", "connectivity_increase_rank"],
    )
    .rename(columns={"connectivity_increase_rank": "connectivity_rank"})
    .drop_duplicates(subset=["catchment_uid"])
)


In [ ]:
# BCR ranks
bcr = pd.read_csv(
    results_path / "catchment_costs_avoided_ead_with_bcr_max.csv",
    usecols=["catchment_uid", "bcr_usd_discounted"],
)
bcr["bcr_usd_discounted"] = pd.to_numeric(bcr["bcr_usd_discounted"], errors="coerce")
bcr["bcr_usd_discounted"].replace([np.inf, -np.inf], np.nan, inplace=True)
bcr["bcr_rank"] = bcr["bcr_usd_discounted"].rank(method="dense", ascending=False).astype("Int64")


In [ ]:
# Merge and compute MCA score (sum of ranks), then sort best-to-worst
mca = bcr.merge(connectivity, on="catchment_uid", how="inner")
mca["mca_score"] = (mca["bcr_rank"] + mca["connectivity_rank"]).astype("Int64")
mca_sorted = (
    mca[["catchment_uid", "bcr_rank", "connectivity_rank", "mca_score"]]
    .sort_values(["mca_score", "catchment_uid"])
    .reset_index(drop=True)
)

In [ ]:
# Save
output_path = results_path / "catchment_mca_ranks_max.csv"
mca_sorted.to_csv(output_path, index=False)
display(mca_sorted)
output_path